In [1]:
import tenpy
from tenpy.models.model import CouplingMPOModel
from tenpy.networks.site import SpinHalfFermionSite
from tenpy.networks.mps import MPS
from tenpy.algorithms import dmrg as tenpy_dmrg

print(f"TeNPy version: {tenpy.__version__}")

TeNPy version: 1.1.0


In [ ]:
# ─── Model definition ─────────────────────────────────────────────────────────
class HubbardSpinDepV(CouplingMPOModel):
    """
    1D Hubbard model with spin-dependent nearest-neighbor repulsion.

    Model parameters:
        t     : float — NN hopping (default 1.0)
        U     : float — on-site Hubbard repulsion 
        Vpp   : float — V^{++} = V^{--}, same-spin NN repulsion 
        Vpm   : float — V^{+-} = V^{-+}, opposite-spin NN repulsion
                        FM condition: Vpm > Vpp
        mu    : float — chemical potential (default 0.0, half filling)
        bc_MPS    : 'infinite' or 'finite'
        L         : unit cell / chain length
    """

    def init_sites(self, model_params):
        cons_N = model_params.get("cons_N", "None")
        cons_Sz = model_params.get("cons_Sz", "None")
        return SpinHalfFermionSite(cons_N = cons_N, cons_Sz = cons_Sz)

    def init_terms(self, model_params):
        t   = model_params.get("t",   1.0)
        U   = model_params.get("U", None)
        Vpp = model_params.get("Vpp", None)   # V^{++}: same spin
        Vpm = model_params.get("Vpm", None)   # V^{+-}: opposite spin
        mu  = model_params.get("mu",  0.0)

        # NN hopping (plus_hc=True adds the Hermitian conjugate automatically)
        self.add_coupling(-t, 0, "Cdu", 0, "Cu", 1, plus_hc=True)
        self.add_coupling(-t, 0, "Cdd", 0, "Cd", 1, plus_hc=True)

        # On-site Hubbard U
        self.add_onsite(U, 0, "NuNd")

        # V^{++}: same-spin NN repulsion  (↑↑ and ↓↓)
        self.add_coupling(Vpp, 0, "Nu", 1, "Nu", 1)
        self.add_coupling(Vpp, 0, "Nd", 1, "Nd", 1)

        # V^{+-}: opposite-spin NN repulsion  (↑↓ and ↓↑)
        self.add_coupling(Vpm, 0, "Nu", 1, "Nd", 1)
        self.add_coupling(Vpm, 0, "Nd", 1, "Nu",   1)

        # Chemical potential
        if abs(mu) > 1e-12:
            self.add_onsite(-mu, 0, "Ntot")

In [13]:
model_params = dict(
        t=1, U=2, Vpp=0.1, Vpm=2, mu=0.0,
        bc_MPS="infinite",
        L=2,
    )
model = HubbardSpinDepV(model_params)
model.lat.N_sites

IndexError: list index out of range

In [3]:
# ─── iDMRG ───────────────────────────────────────────────────────────────────
def run_idmrg(t=1.0, U=4.0, Vpp=0.5, Vpm=1.5, chi_max=300, n_sweeps=12, max_err=1e-5,
              verbose=True):
    """
    Run iDMRG for HubbardSpinDepV in the thermodynamic limit.
    Unit cell L=2 (minimum for a non-trivial iMPS at half filling).
    """
    if verbose:
        print("=" * 65)
        print(f"  t={t}  U={U}  V^{{++}}={Vpp}  V^{{+-}}={Vpm}")
        print(f"  ΔV = V^{{+-}} - V^{{++}} = {Vpm-Vpp:.3f}  (>0 drives FM)")
        print("=" * 65)

    model_params = dict(
        t=t, U=U, Vpp=Vpp, Vpm=Vpm, mu=0.0,
        bc_MPS="infinite",
        L=2,
    )
    model = HubbardSpinDepV(model_params)

    # Half-filling, Sz=0 initial state
    psi = MPS.from_product_state(model.lat.mps_sites(), ["up", "down"] , bc=model.lat.bc_MPS)

    chi_list = {0:chi_max//2, 21: int(3*chi_max/4), 51:chi_max}
    dmrg_params = {
        "trunc_params": {
            "chi_max": chi_max,
            "svd_min": 1e-11,
            "trunc_cut": 1e-8,
        },
        "chi_list":chi_list,
        "mixer": "DensityMatrixMixer",
        "mixer_params": {
            "amplitude": 1e-5,
            "decay": 1.5,
            "disable_after": 60,
        },
        "N_sweeps_check": 1,
        "min_sweeps": 4,
        "max_sweeps": n_sweeps,
        "norm_tol": 1e-6,
        "update_env": 10,
        "start_env": 10,
        'max_E_err': max_err, #precision in energy 
        'max_S_err': max_err, #precision in entropy
    }

    eng = tenpy_dmrg.TwoSiteDMRGEngine(psi, model, dmrg_params)
    E, psi = eng.run()   # E is energy per site for iDMRG

    # if verbose:
    #     _report_idmrg(E, psi)

    return E, psi, model